In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        print(e)

1 Physical GPUs, 1 Logical GPUs


In [ ]:
df = pd.read_csv("hf://datasets/vg055/Rest-Mex2025/Rest-Mex_2025_train.csv")[["Review", "Polarity"]]

In [ ]:
df["Polarity"].value_counts()

,count
Polarity,
5.0,136561
4.0,45034
3.0,15519
2.0,5496
1.0,5441


In [ ]:
sentiment_dict = {
    1: 0,
    2: 0,
    3: 1,
    4: 2,
    5: 2,
}
df["Sentiment"] = df["Polarity"].astype(int).map(lambda x: sentiment_dict[x])
del df["Polarity"]
df.head()

,Review,Sentiment
0,Excelente lugar para comer y pasar una buena n...,2
1,"andar mucho, así que un poco difícil para pers...",2
2,"Es nuestra cuarta visita a Dreams Tulum, elegi...",1
3,"Estando en CancÃºn, fuimos al puerto y tomamos...",2
4,Es un lugar antiguo y por eso me encanto tiene...,2


In [ ]:
df.shape

(208051, 2)

In [ ]:
df["Sentiment"].value_counts()

,count
Sentiment,
2,181595
1,15519
0,10937


In [ ]:
df.iloc[0].Review

'Excelente lugar para comer y pasar una buena noche!!!\nEl servicio es de primera y la comida exquisita!!!'

In [ ]:
df.iloc[0].Review.split(" ")

['Excelente',
 'lugar',
 'para',
 'comer',
 'y',
 'pasar',
 'una',
 'buena',
 'noche!!!\nEl',
 'servicio',
 'es',
 'de',
 'primera',
 'y',
 'la',
 'comida',
 'exquisita!!!']

In [ ]:
words_set = set()
for row in df.iterrows():
  words_set.update(row[1].Review.split(" "))

In [ ]:
len(words_set)

357424

In [ ]:
X = df[["Review"]]
y = df["Sentiment"]

In [ ]:
X_train, X_test_full, y_train, y_test_full = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_test, X_val, y_test, y_val= train_test_split(
    X_test_full, y_test_full, test_size=0.50, random_state=42
)


In [ ]:
X_train.shape, X_test.shape, X_val.shape

((145635, 1), (31208, 1), (31208, 1))

In [ ]:
X_train.head(3)

,Review
192017,Una cena perfecta con una. Vista maravillosa. ...
155911,Mi tercera ocasión que visitó la cadena Dreams...
22182,Si quieres confiablemente buena comida y buen ...


In [ ]:
words_dict = dict()
for row in X_train.iterrows():
  for word in row[1].Review.split(" "):
    words_dict[word] = words_dict.get(word, 0) + 1

In [ ]:
for word in list(words_dict.keys())[:20]:
  print(word, words_dict[word])

Una 5760
cena 3651
perfecta 1261
con 93666
una. 99
Vista 169
maravillosa. 524
Fuimos 3563
en 203087
este 26970
restaurante 16549
nuestros 2318
dos 9896
niños. 490
El 52875
mesero 2026
fue 27986
muy 112730
atento, 827
los 78369


In [ ]:
len(words_set)

357424

In [ ]:
bow = CountVectorizer(
    max_features = 150_000,
    ngram_range = (1, 1),
    token_pattern = r"(?u)\b\w+\b",
    lowercase = True
)

In [ ]:
bow.fit(X_train["Review"])

CountVectorizer(max_features=150000, token_pattern='(?u)\\b\\w+\\b')

In [ ]:
vocabulary = bow.get_feature_names_out()

In [ ]:
len(vocabulary)

94579

In [ ]:
vocabulary[8000:80010]

array(['apreton', 'apretones', 'apretujada', ..., 'senderismo',
       'senderismosituado', 'senderistas'], shape=(72010,), dtype=object)

In [ ]:
X_train_bow = bow.transform(X_train["Review"])
X_test_bow = bow.transform(X_test["Review"])
X_val_bow = bow.transform(X_val["Review"])

In [ ]:
X_train_bow.shape

(145635, 94579)

In [ ]:
with open("temp.txt", "a+") as f:
  f.write(str(X_train_bow[0].toarray().tolist()))

In [ ]:
fila = X_train_bow[0].toarray()
fila[fila > 0], len(fila[fila > 0])

(array([1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1]), 21)

In [ ]:
indices = np.where(fila > 0)[1]
indices

array([ 9995, 17499, 21666, 31299, 32486, 33399, 36764, 41234, 41309,
       54219, 55820, 57410, 60005, 61244, 61731, 66319, 67943, 76409,
       77108, 89635, 92030])

In [ ]:
vocabulary[indices]

array(['atento', 'cena', 'con', 'dos', 'el', 'en', 'este', 'fue',
       'fuimos', 'los', 'maravillosa', 'mesero', 'muy', 'niños',
       'nuestros', 'perfecta', 'platillos', 'restaurante', 'ricos', 'una',
       'vista'], dtype=object)

In [ ]:
X_train.iloc[0].Review

'Una cena perfecta con una. Vista maravillosa. Fuimos en este restaurante con nuestros dos niños. El mesero fue muy atento, los platillos muy ricos.'

In [ ]:
X_train_bow.shape

(145635, 94579)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_bow.shape[1],)),
    tf.keras.layers.Dense(units = 64, activation = "relu"),
    tf.keras.layers.Dense(units = 3, activation = "softmax")
])

model.compile(
    optimizer = "adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train_bow,
    y_train,
    epochs = 5,
    batch_size = 1024,
    validation_data = (X_val_bow, y_val)
)

Epoch 1/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 103s 704ms/step - accuracy: 0.8826 - loss: 0.3889 - val_accuracy: 0.9007 - val_loss: 0.2799
Epoch 2/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 80s 560ms/step - accuracy: 0.9121 - loss: 0.2365 - val_accuracy: 0.9104 - val_loss: 0.2556
Epoch 3/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 80s 559ms/step - accuracy: 0.9292 - loss: 0.1903 - val_accuracy: 0.9075 - val_loss: 0.2577
Epoch 4/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 80s 557ms/step - accuracy: 0.9425 - loss: 0.1584 - val_accuracy: 0.9081 - val_loss: 0.2692
Epoch 5/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 74s 507ms/step - accuracy: 0.9541 - loss: 0.1326 - val_accuracy: 0.9074 - val_loss: 0.2905


In [ ]:
review = "El atole de champaña y el tamal de caviar no eran delicosos, pésimo servicio, no volveria a ir jamás"
review_bow = bow.transform([review])
model.predict(review_bow)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 601ms/step


array([[0.6669178 , 0.15760225, 0.17548002]], dtype=float32)

In [ ]:
text_layer = tf.keras.layers.TextVectorization(
    max_tokens = 60_000,
    output_mode = "int",
    output_sequence_length = 100,
)

text_layer.adapt(X_train["Review"])

In [ ]:
text_layer.vocabulary_size()

60000

In [ ]:
model_token = tf.keras.Sequential([
    text_layer,
    tf.keras.layers.Embedding(60_000, 64),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units = 64, activation = "relu"),
    tf.keras.layers.Dense(units = 3, activation = "softmax")
])

model_token.compile(
    optimizer = "adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model_token.fit(
    X_train["Review"].to_numpy(),
    y_train,
    epochs = 5,
    batch_size = 1024,
    validation_data = (X_val["Review"].to_numpy(), y_val)
)

Epoch 1/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.8763 - loss: 0.3685 - val_accuracy: 0.8983 - val_loss: 0.2926
Epoch 2/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.9097 - loss: 0.2420 - val_accuracy: 0.9072 - val_loss: 0.2587
Epoch 3/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.9350 - loss: 0.1769 - val_accuracy: 0.9071 - val_loss: 0.2741
Epoch 4/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.9707 - loss: 0.0934 - val_accuracy: 0.9017 - val_loss: 0.3357
Epoch 5/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9920 - loss: 0.0332 - val_accuracy: 0.8981 - val_loss: 0.4018


In [ ]:
model_token_conv = tf.keras.Sequential([
    text_layer,
    tf.keras.layers.Embedding(60_000, 1024),
    tf.keras.layers.Conv1D(64, kernel_size = 5, padding = "same", activation = "relu"),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units = 64, activation = "relu"),
    tf.keras.layers.Dense(units = 3, activation = "softmax")
])

model_token_conv.compile(
    optimizer = "adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model_token_conv.fit(
    X_train["Review"].to_numpy(),
    y_train,
    epochs = 5,
    batch_size = 1024,
    validation_data = (X_val["Review"].to_numpy(), y_val)
)

Epoch 1/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 29s 154ms/step - accuracy: 0.8844 - loss: 0.3276 - val_accuracy: 0.9045 - val_loss: 0.2608
Epoch 2/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 20s 142ms/step - accuracy: 0.9201 - loss: 0.2094 - val_accuracy: 0.9072 - val_loss: 0.2544
Epoch 3/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 20s 142ms/step - accuracy: 0.9557 - loss: 0.1229 - val_accuracy: 0.9015 - val_loss: 0.3065
Epoch 4/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 20s 141ms/step - accuracy: 0.9892 - loss: 0.0374 - val_accuracy: 0.9047 - val_loss: 0.5023
Epoch 5/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 21s 148ms/step - accuracy: 0.9985 - loss: 0.0081 - val_accuracy: 0.9007 - val_loss: 0.5811


In [ ]:
review = "El atole de champaña y el tamal de caviar no eran delicosos, pésimo servicio, no volveria a ir jamás"
model_token_conv.predict(np.array([review], dtype=object))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step


array([[0.69672203, 0.13823588, 0.16504204]], dtype=float32)

In [ ]:
review = "El atole de champaña y el tamal de caviar no eran delicosos, pésimo servicio"
model_token_conv.predict(np.array([review], dtype=object))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


array([[0.37820438, 0.33677533, 0.28502026]], dtype=float32)